# Phase P6: Run C — Reversible Max Batch Training
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/05_reversible_max_batch.ipynb)


In [ ]:
import os, sys, json
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/train_with_reversibility' if os.path.exists('/content/train_with_reversibility') else '.')


In [ ]:
from revllm.config import ModelConfig, RunConfig, TrainConfig
from revllm.train import train_run

model_cfg = ModelConfig.from_yaml("configs/model.yaml")
train_cfg = TrainConfig.from_yaml("configs/train.yaml")
run_cfg = RunConfig.from_yaml("configs/runs/C.yaml")

# Load winner variant from pilot
if os.path.exists("results/variant_selection.json"):
    with open("results/variant_selection.json") as f:
        v_data = json.load(f)
        run_cfg.arch = v_data.get("winner", run_cfg.arch)
        run_cfg.h = 0.5 if run_cfg.arch == "midpoint" else 1.0

# Batch size MUST be max safe batch for the reversible variant (B_C)
if os.path.exists("results/batch_probe.json"):
    with open("results/batch_probe.json") as f:
        probes = json.load(f)
        run_cfg.batch_size = probes.get("b_C", run_cfg.batch_size)

drive_dir = "/content/drive/MyDrive/revllm/runs/rev_max" if IN_COLAB and not SMOKE else None
result = train_run(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    run_cfg=run_cfg,
    train_data_path="data/train.bin",
    val_data_path="data/val.bin",
    local_output_dir="results/runs/C",
    drive_output_dir=drive_dir,
    smoke_mode=SMOKE,
)
print("Run C Completed! Final Result:", json.dumps(result, indent=2))
